In [ ]:
import torch
from huggingface_hub import create_repo, upload_folder
from transformers import AutoTokenizer

from models.ema import ExponentialMovingAverage
from models.hf import Proseco, ProsecoConfig

### LLaDA SFT Model

In [ ]:
repo_id = "kuleshov-group/proseco-llada-sft"
create_repo(repo_id, repo_type="model", private=False)

In [ ]:
# TODO: set this path
local_folder_path = "<PATH_TO_LOCAL_FOLDER>"
upload_folder(
    repo_id=repo_id,
    folder_path=local_folder_path,
    commit_message="Upload initial model files"
)

### From Scratch OWT Model

In [ ]:
ProsecoConfig.register_for_auto_class()
Proseco.register_for_auto_class('AutoModelForMaskedLM')

In [ ]:
owt_repo_id = 'kuleshov-group/proseco-owt'
device = 'cuda'
tokenizer = AutoTokenizer.from_pretrained('gpt2', trust_remote_code=True)

In [ ]:
owt_config = ProsecoConfig(
    vocab_size=tokenizer.vocab_size + 1,
    model_length=1024,
    hidden_dim=768,
    cond_dim=128,
    n_blocks=12,
    n_heads=12,
    dropout=0.1,
    time_conditioning=False,
    cfg=False,
    cfg_num_classes=-1,
    return_dict=False
)

In [ ]:
owt_model = Proseco(owt_config)
ema = ExponentialMovingAverage(
    owt_model.backbone.parameters(),
    decay=0.0)

In [ ]:
owt_model.config._name_or_path = owt_repo_id
owt_model.config.auto_map = {
    "AutoConfig": f"{owt_repo_id}--configuraction_proseco.ProsecoConfig",
    "AutoModelForMaskedLM": f"{owt_repo_id}--modeling_proseco.Proseco",
}

In [ ]:
# TODO: set this path
owt_model_ckpt_path = "<PATH_TO_OWT_CHECKPOINT>"
ckpt = torch.load(owt_model_ckpt_path)

In [ ]:
device = "cuda"
ema.load_state_dict(ckpt["ema"])
ema.copy_to(owt_model.backbone.parameters())
owt_model = owt_model.to(device)

In [ ]:
# Confirm EMA params loaded
for c, m in zip(ema.shadow_params, ckpt["ema"]["shadow_params"]):
    if not torch.allclose(c.to(device), m.to(device)):
        print("Issue with EMA!")

for c, m in zip(ema.shadow_params, owt_model.parameters()):
    if not torch.allclose(c.to(device), m.to(device)):
        print("Issue with EMA!")

In [ ]:
owt_model.push_to_hub(owt_repo_id, private=False)

In [ ]:
# # Test model from HF
# owt_model_test = AutoModelForMaskedLM.from_pretrained(owt_repo_id, trust_remote_code=True)
# input_ids = torch.randint(10, size=(2, 10)).to(device)
# model_test = owt_model_test.to(device)
# owt_model_test.eval()
# print(owt_model_test(input_ids, torch.zeros(2,).to(device)).shape)
# print(owt_model_test(input_ids, torch.zeros(2,).to(device)).mean())